# Step 1 — Data Ingestion and Cleaning

Import from SQLite, validate schema, fix negatives, enforce the inventory balance
equation. Implementation in `src/mig_cement/data/`.

In [1]:
import numpy as np
import pandas as pd

from mig_cement.config import settings
from mig_cement.data import load, preprocess, validate

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

## 1. Ingestion

In [2]:
ops = load.load_operations()
sites = load.load_sites()
types = load.load_cement_types()

print(f"Operations  {ops.shape[0]:,} x {ops.shape[1]}")
print(f"Sites       {sites.shape[0]}")
print(f"CementTypes {types.shape[0]} -> {types.cement_type.tolist()}")
print(f"Dates       {ops.date.min().date()} -> {ops.date.max().date()}")

Operations  32,880 x 11
Sites       30
CementTypes 3 -> ['CEM_I', 'CEM_II', 'CEM_III']
Dates       2022-01-01 -> 2024-12-31


In [3]:
panel = load.load_panel()
panel.head(3)

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,North,aggressive
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,North,aggressive
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,North,aggressive


## 2. Schema

In [4]:
print("missing columns:", validate.check_schema(panel) or "none")
print("nulls:", int(panel.isna().sum().sum()))
panel.dtypes.to_frame("dtype").T

missing columns: none
nulls: 0


,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior
dtype,datetime64[ns],object,object,float64,float64,float64,float64,float64,float64,float64,int64,object,object


## 3. Grain

The series key is `site_id`, not `(site_id, cement_type)`.

In [5]:
n_sites, n_dates = panel.site_id.nunique(), panel.date.nunique()
print(f"{n_sites} sites x {n_dates} dates = {n_sites * n_dates:,} | rows = {len(panel):,}")
print("one row per site-day:", (panel.groupby(['site_id','date']).size() == 1).all())
print(panel.cement_type.value_counts(normalize=True).round(3).to_dict())

30 sites x 1096 dates = 32,880 | rows = 32,880
one row per site-day: True
{'CEM_II': 0.338, 'CEM_I': 0.332, 'CEM_III': 0.331}


In [6]:
def continuity(df, keys):
    d = df.sort_values(keys + ["date"])
    prev = d.groupby(keys).closing_inventory_tonnes.shift(1)
    return ((d.opening_inventory_tonnes - prev).abs() < 0.01)[prev.notna()].mean()

print(f"site grain      {continuity(panel, ['site_id']):.1%}")
print(f"site-type grain {continuity(panel, ['site_id','cement_type']):.1%}")

site grain      100.0%
site-type grain 42.7%


In [7]:
preprocess.check_calendar(panel)

{'n_sites': 30,
 'n_dates': 1096,
 'complete': True,
 'duplicated_site_days': 0,
 'multi_type_site_days': 0}

## 4. Raw quality

In [8]:
pd.Series(validate.validate_raw(panel)).to_frame("count")

,count
missing_columns,0
duplicate_keys,0
negative_values,0
balance_breaches,2
capacity_breaches,11439
ledger_discontinuities,0


In [9]:
# negatives: none present, so fix_negatives is a guard
panel[["consumed_tonnes","deliveries_tonnes",
       "opening_inventory_tonnes","closing_inventory_tonnes"]].min().to_frame("min")

,min
consumed_tonnes,0.0
deliveries_tonnes,0.0
opening_inventory_tonnes,0.0
closing_inventory_tonnes,0.0


In [10]:
validate.check_balance(panel)[["date","site_id","opening_inventory_tonnes",
    "deliveries_tonnes","consumed_tonnes","closing_inventory_tonnes"]]

,date,site_id,opening_inventory_tonnes,deliveries_tonnes,consumed_tonnes,closing_inventory_tonnes
25208,2022-01-01,SITE_024,73.23,36.61,64.29,45.56
28496,2022-01-01,SITE_027,62.79,13.10,12.38,63.52


In [11]:
panel.assign(over=panel.closing_inventory_tonnes > panel.silo_capacity).groupby("behavior").agg(
    pct_over_capacity=("over", lambda s: round(100 * s.mean(), 1)),
    mean_closing_t=("closing_inventory_tonnes", lambda s: round(s.mean(), 1)),
    max_closing_t=("closing_inventory_tonnes", "max"),
)

,pct_over_capacity,mean_closing_t,max_closing_t
behavior,,,
aggressive,0.0,13.5,266.10
chaotic,22.2,180.0,922.26
conservative,98.7,10079.8,20658.87


In [12]:
censored = panel.consumed_tonnes < panel.planned_pour_tonnes - 1e-6
print(f"consumption below plan: {censored.mean():.1%}")
print(f"unmet demand: {(panel.planned_pour_tonnes - panel.consumed_tonnes).clip(lower=0).sum():,.0f} t")

consumption below plan: 39.7%
unmet demand: 229,393 t


## 5. Cleaning

Deliveries capped at available headroom; excess recorded as rejected.

In [13]:
clean, report = preprocess.build_clean_panel(panel, mode="cap_deliveries")
print(f"{len(clean):,} rows, {clean.shape[1]} columns")

32,880 rows, 22 columns


## 6. Validation and output

In [14]:
validate.validate_clean(clean)
print("passed")

passed


In [15]:
dest = settings.interim_dir / "operations_clean.parquet"
dest.parent.mkdir(parents=True, exist_ok=True)
clean.to_parquet(dest, index=False)
print(f"wrote {len(clean):,} rows -> {dest.name}")

wrote 32,880 rows -> operations_clean.parquet
